In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
model = ChatGroq(model="llama-3.1-8b-instant")

In [3]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [4]:
graph = StateGraph(JokeState)

In [5]:
def generate_joke(state: JokeState):
    prompt = f"generate a joke on the topic {state['topic']}"
    joke = model.invoke(prompt).content
    return {"joke": joke}

In [6]:
def generate_explanation(state: JokeState):
    prompt = f"Write an explanation for the joke :- \n {state['joke']}"
    explanation = model.invoke(prompt).content
    return {"explanation": explanation}

In [7]:
graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

In [8]:
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

In [9]:
checkpointer = InMemorySaver()

In [10]:
workflow = graph.compile(checkpointer=checkpointer)

In [11]:
config1 = {"configurable": {"thread_id": 1}}
workflow.invoke({"topic": "pizza"}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.',
 'explanation': 'The joke is a play on words, using a pun to create humor. Here\'s a breakdown of the explanation:\n\n- The setup of the joke, "Why did the pizza go to the doctor?", is a common format for a joke. It asks a question and sets up a situation that is expected to lead to a punchline.\n- The punchline, "Because it was feeling a little crusty", is a clever use of wordplay. In this context, "crusty" has a double meaning:\n  - A crust is a part of a pizza, specifically the outer layer made of dough.\n  - "Crusty" is also an idiomatic expression meaning feeling or looking a bit rough, grumpy, or irritated.\n- The humor comes from the unexpected twist on the usual meaning of "crusty". It takes the listener a moment to process that the joke is not just about the pizza\'s physical appearance but also about its emotional state.\n- The punchline relies on the listener\'s pri

In [12]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke is a play on words, using a pun to create humor. Here\'s a breakdown of the explanation:\n\n- The setup of the joke, "Why did the pizza go to the doctor?", is a common format for a joke. It asks a question and sets up a situation that is expected to lead to a punchline.\n- The punchline, "Because it was feeling a little crusty", is a clever use of wordplay. In this context, "crusty" has a double meaning:\n  - A crust is a part of a pizza, specifically the outer layer made of dough.\n  - "Crusty" is also an idiomatic expression meaning feeling or looking a bit rough, grumpy, or irritated.\n- The humor comes from the unexpected twist on the usual meaning of "crusty". It takes the listener a moment to process that the joke is not just about the pizza\'s physical appearance but also about its emotional state.\n- The punchline relies on 

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke is a play on words, using a pun to create humor. Here\'s a breakdown of the explanation:\n\n- The setup of the joke, "Why did the pizza go to the doctor?", is a common format for a joke. It asks a question and sets up a situation that is expected to lead to a punchline.\n- The punchline, "Because it was feeling a little crusty", is a clever use of wordplay. In this context, "crusty" has a double meaning:\n  - A crust is a part of a pizza, specifically the outer layer made of dough.\n  - "Crusty" is also an idiomatic expression meaning feeling or looking a bit rough, grumpy, or irritated.\n- The humor comes from the unexpected twist on the usual meaning of "crusty". It takes the listener a moment to process that the joke is not just about the pizza\'s physical appearance but also about its emotional state.\n- The punchline relies on

In [14]:
config2 = {"configurable": {"thread_id": 2}}
workflow.invoke({"topic": "pasta"}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta go to therapy? \n\nBecause it was feeling a little "twisted."',
 'explanation': 'The joke is a play on words, using a pun to create humor. \n\nIn this joke, "twisted" has a double meaning. On one hand, pasta is a type of food that can be twisted into different shapes, like fusilli or corkscrew pasta. \n\nOn the other hand, "feeling a little \'twisted\'" is an idiomatic expression that means feeling emotionally or mentally unstable or having a peculiar feeling. It\'s often used to describe someone who is experiencing anxiety, stress, or other mental health issues.\n\nThe joke is funny because it takes the literal meaning of "twisted" (referring to pasta shapes) and interprets it in a more figurative sense (referring to emotional distress), creating a clever connection between the setup and the punchline. This wordplay is what makes the joke amusing and engaging.'}

In [15]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to therapy? \n\nBecause it was feeling a little "twisted."', 'explanation': 'The joke is a play on words, using a pun to create humor. \n\nIn this joke, "twisted" has a double meaning. On one hand, pasta is a type of food that can be twisted into different shapes, like fusilli or corkscrew pasta. \n\nOn the other hand, "feeling a little \'twisted\'" is an idiomatic expression that means feeling emotionally or mentally unstable or having a peculiar feeling. It\'s often used to describe someone who is experiencing anxiety, stress, or other mental health issues.\n\nThe joke is funny because it takes the literal meaning of "twisted" (referring to pasta shapes) and interprets it in a more figurative sense (referring to emotional distress), creating a clever connection between the setup and the punchline. This wordplay is what makes the joke amusing and engaging.'}, next=(), config={'configurable': {'thread_id': '2', 'chec

In [16]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to therapy? \n\nBecause it was feeling a little "twisted."', 'explanation': 'The joke is a play on words, using a pun to create humor. \n\nIn this joke, "twisted" has a double meaning. On one hand, pasta is a type of food that can be twisted into different shapes, like fusilli or corkscrew pasta. \n\nOn the other hand, "feeling a little \'twisted\'" is an idiomatic expression that means feeling emotionally or mentally unstable or having a peculiar feeling. It\'s often used to describe someone who is experiencing anxiety, stress, or other mental health issues.\n\nThe joke is funny because it takes the literal meaning of "twisted" (referring to pasta shapes) and interprets it in a more figurative sense (referring to emotional distress), creating a clever connection between the setup and the punchline. This wordplay is what makes the joke amusing and engaging.'}, next=(), config={'configurable': {'thread_id': '2', 'che

Time Travel

In [18]:
workflow.get_state({"configurable": {"thread_id": 1, "checkpoint_id": "1f0bbd63-37a4-6c01-8000-b7ceda136852"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f0bbd63-37a4-6c01-8000-b7ceda136852'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2025-11-07T12:35:10.953350+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0bbd63-3798-6bd5-bfff-281d218fe598'}}, tasks=(PregelTask(id='fecca4d7-067c-2484-4779-f0ed04d445b0', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.'}),), interrupts=())

In [19]:
workflow.invoke(None, {"configurable": {"thread_id": 1, "checkpoint_id": "1f0bbd63-37a4-6c01-8000-b7ceda136852"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.',
 'explanation': 'The joke is a play on words. It\'s a pun, which is a form of wordplay that exploits multiple meanings of words or phrases. \n\nIn this case, the joke is referencing both the physical properties of a pizza (the crust is a thick outer layer of bread that makes up the base of the pizza) and a common expression for being in a bad mood or feeling unwell (someone who is feeling a little "crusty" might be grumpy or irritable).\n\nThe joke relies on the listener to make this double meaning connection between the literal description of a pizza and the idiomatic expression for being in a bad mood. The punchline, "it was feeling a little crusty," creates a humorous connection between the pizza\'s physical properties and its emotional state, making it a lighthearted and amusing joke.'}

In [20]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke is a play on words. It\'s a pun, which is a form of wordplay that exploits multiple meanings of words or phrases. \n\nIn this case, the joke is referencing both the physical properties of a pizza (the crust is a thick outer layer of bread that makes up the base of the pizza) and a common expression for being in a bad mood or feeling unwell (someone who is feeling a little "crusty" might be grumpy or irritable).\n\nThe joke relies on the listener to make this double meaning connection between the literal description of a pizza and the idiomatic expression for being in a bad mood. The punchline, "it was feeling a little crusty," creates a humorous connection between the pizza\'s physical properties and its emotional state, making it a lighthearted and amusing joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns

Using Time Travel(Persistence) Updating State

In [22]:
workflow.update_state({"configurable": {"thread_id": 1, "checkpoint_id": "1f0bbd63-37a4-6c01-8000-b7ceda136852", "checkpoint_ns": ""}}, {"topic": "samosa"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0bbd7f-b34a-6e0e-8001-8e2dd9194ad4'}}

In [23]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0bbd7f-b34a-6e0e-8001-8e2dd9194ad4'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2025-11-07T12:47:55.538156+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0bbd63-37a4-6c01-8000-b7ceda136852'}}, tasks=(PregelTask(id='058d172e-9b3f-63b5-0936-6ed5c97f8cc7', name='generate_explanation', path=('__pregel_pull', 'generate_explanation'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke is a play on words. It\'s a pun, which is a form of wordplay that exploits multiple meanings of words or phrase

In [28]:
workflow.invoke(None, {"configurable": {"thread_id": 1, "checkpoint_id": "1f0bbd7f-b34a-6e0e-8001-8e2dd9194ad4"}})

{'topic': 'samosa',
 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.',
 'explanation': 'The joke "Why did the pizza go to the doctor? Because it was feeling a little crusty" is a play on words. It uses a pun to create the humor.\n\nA "crusty" has a double meaning here:\n\n1. In a literal sense, a pizza has a crust, which is its outer layer made of bread.\n2. In an idiomatic sense, "feeling a little crusty" is a common phrase used to describe someone who is feeling irritable, grumpy, or a bit rough around the edges.\n\nThe joke relies on this wordplay to create the humor. The punchline "it was feeling a little crusty" is a clever and unexpected twist on the expected answer, which would typically be a health-related reason for going to the doctor. The joke is a lighthearted and silly way to poke fun at the idea of a pizza needing medical attention, and the pun on "crusty" is the key to its humor.'}

In [29]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke "Why did the pizza go to the doctor? Because it was feeling a little crusty" is a play on words. It uses a pun to create the humor.\n\nA "crusty" has a double meaning here:\n\n1. In a literal sense, a pizza has a crust, which is its outer layer made of bread.\n2. In an idiomatic sense, "feeling a little crusty" is a common phrase used to describe someone who is feeling irritable, grumpy, or a bit rough around the edges.\n\nThe joke relies on this wordplay to create the humor. The punchline "it was feeling a little crusty" is a clever and unexpected twist on the expected answer, which would typically be a health-related reason for going to the doctor. The joke is a lighthearted and silly way to poke fun at the idea of a pizza needing medical attention, and the pun on "crusty" is the key to its humor.'}, next=(), config={'configurab